# SHAP Explainability — UK CNP Fraud Detection

This notebook explains the XGBoost fraud scoring model using SHAP (SHapley Additive exPlanations).

**Outputs:**
- Global feature importance (beeswarm + bar)
- Single-transaction waterfall explanation
- Dependence plots for top features

Run from the project root: `jupyter notebook notebooks/shap_explainability.ipynb`

In [ ]:
import sys
sys.path.insert(0, '..')  # make src/ importable when running from notebooks/

import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from pathlib import Path

shap.initjs()

## 1. Load model and data

In [ ]:
PROJECT_ROOT = Path('..').resolve()

pipeline = joblib.load(PROJECT_ROOT / 'models' / 'pipeline.pkl')

FEATURE_COLS = [
    'amount_gbp', 'hour_of_day', 'day_of_week', 'card_avg_amount_30d',
    'card_txn_count_1h', 'card_amount_sum_1h', 'card_txn_count_24h',
    'card_amount_sum_24h', 'merch_txn_count_1h', 'time_since_last_card_txn_sec',
    'amount_to_card_avg_ratio', 'log_amount',
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'merchant_category',
]
TARGET_COL = 'is_fraud'

df = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'transactions_featured.csv')
print(f'Loaded {len(df):,} rows — fraud rate: {df[TARGET_COL].mean():.4%}')

In [ ]:
# Temporal split mirrors train.py — use test set only
df = df.sort_values('timestamp').reset_index(drop=True)
split = int(len(df) * 0.8)
test_df = df.iloc[split:].copy()

X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

print(f'Test set: {len(X_test):,} rows — fraud rate: {y_test.mean():.4%}')

In [ ]:
# Extract preprocessor and classifier from the sklearn Pipeline
preprocessor = pipeline.named_steps['preprocessor']
xgb_model    = pipeline.named_steps['classifier']

# Transform test data — SHAP needs the numeric array that XGBoost actually sees
X_test_transformed = preprocessor.transform(X_test)

# Recover feature names after OrdinalEncoding
# The ColumnTransformer passes numeric cols through and encodes categoricals
num_features = [c for c in FEATURE_COLS if c != 'merchant_category']
feature_names = num_features + ['merchant_category']

print('Feature matrix shape:', X_test_transformed.shape)

## 2. Compute SHAP values

`TreeExplainer` uses the tree structure to compute exact Shapley values — far faster than model-agnostic alternatives.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)

# Use a 2000-row sample for plotting speed (full test set is ~120k rows)
sample_idx = np.random.default_rng(42).choice(len(X_test_transformed), size=2000, replace=False)
X_sample = X_test_transformed[sample_idx]
y_sample = y_test.iloc[sample_idx].values

shap_values = explainer(X_sample)
print('SHAP values shape:', shap_values.shape)

## 3. Global importance — beeswarm plot

Each dot is one transaction. Position on x-axis = SHAP value (impact on log-odds of fraud). Colour = feature value (red = high, blue = low).

In [ ]:
shap_values.feature_names = feature_names

plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP Beeswarm — Global Feature Impact (test set sample, n=2000)', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/shap_beeswarm.png')

## 4. Global importance — mean |SHAP| bar chart

In [ ]:
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title('Mean |SHAP| — Average Feature Contribution to Fraud Score', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/shap_bar.png')

## 5. Single-transaction waterfall — fraud case

Picks the highest-probability fraud transaction in the sample and shows exactly which features drove that score.

In [ ]:
# Find the sample transaction with highest predicted fraud probability
probs = xgb_model.predict_proba(X_sample)[:, 1]
fraud_idx = np.argmax(probs)

print(f'Transaction index in sample: {fraud_idx}')
print(f'Predicted fraud probability: {probs[fraud_idx]:.4f}')
print(f'Actual label: {"FRAUD" if y_sample[fraud_idx] == 1 else "LEGIT"}')

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[fraud_idx], max_display=15, show=False)
plt.title(f'Waterfall — Highest-Scored Transaction (p={probs[fraud_idx]:.4f})', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'shap_waterfall_fraud.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/shap_waterfall_fraud.png')

## 6. Single-transaction waterfall — legitimate case

Counterpart: a confidently-scored legitimate transaction.

In [ ]:
legit_idx = np.argmin(probs)

print(f'Transaction index in sample: {legit_idx}')
print(f'Predicted fraud probability: {probs[legit_idx]:.4f}')
print(f'Actual label: {"FRAUD" if y_sample[legit_idx] == 1 else "LEGIT"}')

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[legit_idx], max_display=15, show=False)
plt.title(f'Waterfall — Lowest-Scored Transaction (p={probs[legit_idx]:.4f})', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'shap_waterfall_legit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/shap_waterfall_legit.png')

## 7. Dependence plot — amount_to_card_avg_ratio

Shows how the strongest individual feature interacts with the model score.

In [ ]:
ratio_idx = feature_names.index('amount_to_card_avg_ratio')

plt.figure(figsize=(9, 5))
shap.plots.scatter(
    shap_values[:, ratio_idx],
    color=shap_values,
    show=False
)
plt.title('SHAP Dependence — amount_to_card_avg_ratio', pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'shap_dependence_amount_ratio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/shap_dependence_amount_ratio.png')

## 8. Summary table — top features by mean |SHAP|

In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
importance_df = (
    pd.DataFrame({'feature': feature_names, 'mean_abs_shap': mean_abs_shap})
    .sort_values('mean_abs_shap', ascending=False)
    .reset_index(drop=True)
)
importance_df.index += 1  # rank from 1
importance_df.style.format({'mean_abs_shap': '{:.4f}'}).bar(subset=['mean_abs_shap'], color='#d65f5f')